# Chapter 3: Data Models and Query Languages

*From: Designing Data-Intensive Applications*

---

## TL;DR

- **Data models shape what you can think about.** The chapter compares five families of models — relational, document, graph (property + triple), event log, and DataFrames — and shows that each answers a different "what kinds of relationships dominate?" question.
- **The core trade-off is normalized vs. denormalized**, which is really a trade-off between write amplification and read amplification. X/Twitter's materialized timeline is the canonical real-world compromise: denormalize the list of post IDs, but hydrate content and profile pics at read time because they change too often.
- **Many-to-many is the axis that distinguishes model choice.** One-to-few fits documents; many-to-many fits relational joins; deeply variable traversal fits graphs; heterogeneous event history fits event sourcing.
- **Query languages are declarative tools with different "shapes":** SQL (relations), Cypher/SPARQL (pattern-match traversal), Datalog (recursive rules), GraphQL (client-driven JSON shape), MongoDB aggregation pipeline (stage-by-stage document ops).
- **Analytics flips the write/read bias**: star and snowflake schemas happily denormalize because facts are immutable and reads dominate. Event sourcing goes further — events are the source of truth, and you derive as many read-optimized materialized views as you need.

---

## Requirements

Unlike an end-to-end "design Twitter" chapter, Chapter 3 is a **decision framework**. The functional and non-functional lenses translate to the *criteria a data model must satisfy*:

### Functional Requirements (what a data model must let you express)

- **Entities and attributes** — persist application domain objects (users, posts, sales).
- **One-to-many relationships** — a profile with many positions; a user with many comments.
- **Many-to-one and many-to-many** — users → regions, people → organizations, products in orders.
- **Variable-depth traversal** — "all locations inside the US," "friends of friends," "ancestors in a hierarchy."
- **Heterogeneous entities in one graph** — people, places, events, check-ins all in the same dataset.
- **Client-shaped responses** — a mobile app wants a tailored JSON tree in one round trip.
- **Immutable history** — audit logs, GDPR-sensitive data, reversible bookings.

### Non-Functional Requirements (what the workload demands)

- **Write latency / throughput** — normalized data is cheap to write; denormalized and materialized views amplify writes.
- **Read latency / throughput** — reads may dominate (analytics, timelines), making denormalization worth its cost.
- **Schema evolvability** — schema-on-read vs schema-on-write; ability to add fields without table-rewrite migrations.
- **Consistency** — denormalized copies risk divergence; atomicity across multiple documents is not universal.
- **Query expressiveness vs safety** — GraphQL deliberately forbids recursion and arbitrary search because callers are untrusted.
- **Storage footprint** — materialized timelines, one-big-table warehouses, and hydrated responses trade disk for speed.

---

## The Data-Model Landscape

```mermaid
graph TD
  App[Application Domain Objects]
  App --> Rel[Relational Model<br/>tables, rows, FKs]
  App --> Doc[Document Model<br/>JSON trees]
  App --> PG[Property Graph<br/>vertices + edges + props]
  App --> TS[Triple Store / RDF<br/>subject predicate object]
  App --> EL[Event Log<br/>append-only, immutable]
  App --> DF[DataFrames / Arrays<br/>columnar, matrix]

  Rel -->|SQL, joins| Q1[OLTP + star schemas]
  Doc -->|Mongo aggregation, JSONPath| Q2[self-contained trees]
  PG -->|Cypher| Q3[variable-length traversal]
  TS -->|SPARQL, Datalog| Q4[recursive rules, linked data]
  EL -->|CQRS views| Q5[audit + rebuildable reads]
  DF -->|pandas, Spark| Q6[analytics + ML feature prep]

  classDef pick fill:#eef,stroke:#55a,stroke-width:1px
  class Rel,Doc,PG,TS,EL,DF pick
```

The rest of this notebook walks through each model, the real-world trade-offs that drive the choice, and concrete numbers for the two most commonly asked questions: *"how expensive is a join at read time?"* and *"how expensive is fanning out a denormalized copy at write time?"*

---

## Relational vs. Document: the LinkedIn Example

The classic example is a resume / LinkedIn profile: one user has many positions, many schooling periods, many contact entries. That's a tree of one-to-many relationships.

### Relational schema

```mermaid
erDiagram
  USERS ||--o{ POSITIONS : has
  USERS ||--o{ EDUCATION : has
  USERS ||--o{ CONTACT_INFO : has
  USERS {
    int user_id PK
    text first_name
    text last_name
    text headline
    int region_id FK
  }
  POSITIONS {
    int position_id PK
    int user_id FK
    text job_title
    text organization
  }
  EDUCATION {
    int edu_id PK
    int user_id FK
    text school_name
    int start_year
    int end_year
  }
  CONTACT_INFO {
    int contact_id PK
    int user_id FK
    text channel
    text value
  }
```

### Equivalent JSON document

```json
{
  "user_id": 251,
  "first_name": "Barack",
  "last_name": "Obama",
  "headline": "Former President of the United States of America",
  "region_id": "us:91",
  "positions": [
    {"job_title": "President", "organization": "United States of America"},
    {"job_title": "US Senator (D-IL)", "organization": "United States Senate"}
  ],
  "education": [
    {"school": "Harvard University", "start": 1988, "end": 1991},
    {"school": "Columbia University", "start": 1981, "end": 1983}
  ],
  "contact_info": {"website": "https://barackobama.com"}
}
```

**Locality argument for documents:** if you always render the whole profile, one disk read beats a 4-way join. **Counter-argument for relational:** the moment many people work at the same `organization`, or you want the org logo, duplication kicks in and the document model creaks.

Quick numeric feel for the locality advantage:

In [ ]:
# How much does "fetch a whole profile" cost in the two models?
# Assume a simple cost model: one random seek dominates; sequential bytes are ~free.

RANDOM_SEEK_US = 100  # ~0.1 ms on SSD to locate a page; representative order of magnitude

# Relational: one seek per table (users + positions + education + contact_info)
tables_touched = 4
relational_latency_us = tables_touched * RANDOM_SEEK_US

# Document: one seek, read the entire blob
document_latency_us = 1 * RANDOM_SEEK_US

print(f"Relational profile fetch: ~{relational_latency_us} us ({tables_touched} index lookups)")
print(f"Document profile fetch:   ~{document_latency_us} us (1 lookup)")
print(f"Locality speedup:         {relational_latency_us / document_latency_us:.1f}x")

# But the locality advantage disappears if the doc is huge and you only need a slice.
full_doc_bytes = 500_000       # 500 KB bloated profile
needed_slice_bytes = 2_000     # just the headline + name
waste_ratio = full_doc_bytes / needed_slice_bytes
print(
    f"\nIf the document is {full_doc_bytes/1024:.0f} KB but you only need"
    f" {needed_slice_bytes} bytes, you pay {waste_ratio:.0f}x extra I/O"
    " because the doc is stored as one blob."
)

The point: *document locality pays off only when you actually read most of the document.* Sub-document access patterns and large documents erase the win.

---

## Normalization, Denormalization, and the Cost of Joins

The universal trade-off in one sentence: **normalized data is cheap to write and expensive to read; denormalized data is cheap to read and expensive to keep consistent.**

### The N+1 query problem (ORM hazard)

You fetch N comments, then do N separate lookups to resolve each author's name. That's N+1 round trips instead of 1 join.

In [ ]:
# Quantify the N+1 query problem for a page of comments
N_comments = 50          # comments on one post
per_query_latency_ms = 2 # median DB round trip

# Naive ORM (N+1): one query for the comments, then one per author
n_plus_one_latency_ms = (N_comments + 1) * per_query_latency_ms

# Handwritten SQL with JOIN: one query that returns comment + author name
join_latency_ms = 1 * per_query_latency_ms

print(f"N+1 approach:  {N_comments + 1} queries -> {n_plus_one_latency_ms} ms")
print(f"Joined query:  1 query             -> {join_latency_ms} ms")
print(f"Speedup from avoiding N+1: {n_plus_one_latency_ms / join_latency_ms:.0f}x")

# And scale it to a busy page with 1000 concurrent users:
concurrent_users = 1000
queries_per_second_naive = concurrent_users * (N_comments + 1)
queries_per_second_join  = concurrent_users * 1
print(
    f"\nAt {concurrent_users} concurrent page loads:"
    f" {queries_per_second_naive:,} qps (N+1) vs {queries_per_second_join:,} qps (join)."
)

### Case study: X/Twitter's materialized timeline

The home-timeline join between `posts` and `follows` is too expensive at read time for celebrity accounts, so each follower has a **precomputed, materialized timeline**. But X famously stores only **post IDs**, not post content, because likes and profile pictures change too often to duplicate into millions of timelines.

```mermaid
flowchart LR
  subgraph Write[Write path - post fan-out]
    P[User posts tweet] --> F{Get followers}
    F --> I1[Insert post_id into<br/>follower 1's timeline]
    F --> I2[Insert post_id into<br/>follower 2's timeline]
    F --> IN[...follower N's timeline]
  end

  subgraph Read[Read path - hydrate]
    R[User opens home] --> TL[Read post_ids<br/>from own timeline]
    TL --> HP[Hydrate: fetch post text + likes<br/>for each id]
    TL --> HU[Hydrate: fetch sender profile<br/>pic + username]
    HP --> RND[Render feed]
    HU --> RND
  end
```

Back-of-envelope for the storage cost of materialized timelines:

In [ ]:
# X / Twitter timeline sizing: why store IDs only, not content?
TIMELINE_LEN = 1000             # posts cached per user's home timeline
FOLLOWERS = 200_000_000         # target active followers (feed recipients)

# Option A: denormalize full post content
avg_post_bytes = 280            # just the text
avg_profile_pic_url = 100
bytes_per_entry_content = avg_post_bytes + avg_profile_pic_url

# Option B: denormalize IDs only (what X actually does)
post_id_bytes = 8
sender_id_bytes = 8
flags_bytes = 4                  # repost / reply markers
bytes_per_entry_ids = post_id_bytes + sender_id_bytes + flags_bytes

storage_content_tb = FOLLOWERS * TIMELINE_LEN * bytes_per_entry_content / 1e12
storage_ids_tb     = FOLLOWERS * TIMELINE_LEN * bytes_per_entry_ids     / 1e12

print(f"Option A (full content denormalized): {storage_content_tb:,.1f} TB")
print(f"Option B (IDs only, hydrate on read):  {storage_ids_tb:,.1f} TB")
print(f"Storage reduction from ID-only:        {storage_content_tb/storage_ids_tb:.1f}x")

# Write amplification: a popular user with many followers
celebrity_followers = 100_000_000
writes_to_propagate_one_tweet = celebrity_followers
print(
    f"\nPopular-account fan-out: 1 tweet -> {writes_to_propagate_one_tweet:,}"
    " writes across follower timelines."
)

# And if profile pic changes and we had denormalized it:
# we'd have to rewrite that user's avatar into every timeline entry they appear in.
appearances_per_celebrity = celebrity_followers * 50   # 50 posts each in 100M timelines
print(
    f"Profile-pic change for that celebrity would require"
    f" ~{appearances_per_celebrity:,} row rewrites if the pic were denormalized."
)
print("That's why profile pics stay normalized and are hydrated at read time.")

The pattern: *denormalize the parts that change slowly; keep hot, fast-changing fields normalized and pay the hydration cost on read.*

### Normalized relational join vs. document `$lookup`

```sql
-- Relational: resolve region_id to region_name via JOIN
SELECT users.*, regions.region_name
FROM users
JOIN regions ON users.region_id = regions.id
WHERE users.id = 251;
```

```javascript
// MongoDB aggregation pipeline: equivalent $lookup
db.users.aggregate([
  { $match: { _id: 251 } },
  { $lookup: {
      from: "regions",
      localField: "region_id",
      foreignField: "_id",
      as: "region"
  }}
])
```

Document databases can do joins — they've grown in that direction — but the syntax and ergonomics still favor self-contained trees.

---

## Star and Snowflake Schemas for Analytics

Once you leave OLTP and enter the warehouse, the write/read balance flips: facts are immutable, loaded in bulk, and queried read-heavy. Normalization for consistency matters less; denormalization for query simplicity matters a lot.

```mermaid
graph TD
  FS[fact_sales<br/>row = one line item<br/>~10<sup>10</sup> rows]
  DP[dim_product<br/>SKU, brand, category,<br/>fat content, pkg size]
  DS[dim_store<br/>location, size,<br/>services, opened]
  DC[dim_customer<br/>name, loyalty tier,<br/>region]
  DD[dim_date<br/>day, month, quarter,<br/>is_holiday]
  DP --> FS
  DS --> FS
  DC --> FS
  DD --> FS

  classDef fact fill:#fde,stroke:#a33,stroke-width:2px
  classDef dim  fill:#def,stroke:#37a,stroke-width:1px
  class FS fact
  class DP,DS,DC,DD dim
```

**Snowflake** schemas push dimensions further into sub-dimensions (e.g., `dim_product` references `dim_brand` and `dim_category`). **One Big Table (OBT)** goes the other way: inline everything into the fact row.

Sizing a fact table for a mid-size retailer:

In [ ]:
# Star schema sizing for a grocery retailer
stores = 1_000
items_per_transaction = 8
transactions_per_store_per_day = 2_500
days_retained = 365 * 5           # 5 years of history

fact_rows_per_day = stores * transactions_per_store_per_day * items_per_transaction
fact_rows_total = fact_rows_per_day * days_retained

# Fact rows are narrow: FKs + a few measures
bytes_per_fact_row = (
    4    # date_key
    + 4  # store_key
    + 4  # product_key
    + 4  # customer_key
    + 4  # promotion_key
    + 4  # quantity
    + 8  # net_price (float)
    + 8  # discount (float)
    + 8  # cost (float)
)

fact_bytes_total = fact_rows_total * bytes_per_fact_row
print(f"Fact rows/day:    {fact_rows_per_day:>15,}")
print(f"Fact rows total:  {fact_rows_total:>15,}")
print(f"Fact bytes:       {fact_bytes_total/1e12:>15,.2f} TB")

# Dimension tables are tiny in comparison
dim_product_rows, dim_product_bytes_per_row = 500_000, 2_000   # wide (SKU, desc, brand, ...)
dim_store_rows, dim_store_bytes_per_row     = 1_000,   4_000
dim_customer_rows, dim_customer_bytes_per_row = 20_000_000, 500
dim_date_rows, dim_date_bytes_per_row       = 365 * 20, 200

dim_bytes = (
    dim_product_rows * dim_product_bytes_per_row
    + dim_store_rows * dim_store_bytes_per_row
    + dim_customer_rows * dim_customer_bytes_per_row
    + dim_date_rows * dim_date_bytes_per_row
)
print(f"Dimension bytes:  {dim_bytes/1e9:>15,.2f} GB")
print(
    f"Fact:dim ratio:   {fact_bytes_total / dim_bytes:>15,.0f}x"
    " — storage is dominated by the fact table, which is why denormalizing"
    " dimensions onto the fact row (OBT) costs a *lot*."
)

---

## Graph Data Models: When Many-to-Many Dominates

### Property graph (Cypher)

The running example: Lucy (Person) was BORN_IN Idaho (Location); Idaho is WITHIN USA; USA is WITHIN North America. Each vertex carries properties, each edge has a label.

```mermaid
graph LR
  Lucy((Lucy<br/>Person)) -- BORN_IN --> Idaho((Idaho<br/>state))
  Lucy -- LIVES_IN --> London((London<br/>city))
  Idaho -- WITHIN --> USA((USA<br/>country))
  USA -- WITHIN --> NAm((North America<br/>continent))
  London -- WITHIN --> UK((UK<br/>country))
  UK -- WITHIN --> Europe((Europe<br/>continent))

  classDef person fill:#fde,stroke:#a33
  classDef loc    fill:#def,stroke:#37a
  class Lucy person
  class Idaho,USA,NAm,London,UK,Europe loc
```

The "US-to-Europe emigrants" query is **4 lines in Cypher**:

```cypher
MATCH
  (person)-[:BORN_IN]->()-[:WITHIN*0..]->(:Location {name: 'United States'}),
  (person)-[:LIVES_IN]->()-[:WITHIN*0..]->(:Location {name: 'Europe'})
RETURN person.name
```

The `*0..` pattern is variable-length traversal — "follow a WITHIN edge zero or more times." The equivalent SQL `WITH RECURSIVE` version is **31 lines** (see the chapter).

### Triple stores (SPARQL and Datalog)

A triple store reduces the graph to `(subject, predicate, object)` facts.

```turtle
@prefix : <urn:example:>.
_:lucy     a       :Person ;  :name "Lucy" ;   :bornIn _:idaho .
_:idaho    a       :Location; :name "Idaho" ;  :type "state" ;   :within _:usa .
_:usa      a       :Location; :name "United States" ; :type "country" ; :within _:namerica .
_:namerica a       :Location; :name "North America" ; :type "continent" .
```

The same emigration query in SPARQL:

```sparql
PREFIX : <urn:example:>
SELECT ?personName WHERE {
  ?person :name ?personName .
  ?person :bornIn / :within* / :name "United States" .
  ?person :livesIn / :within* / :name "Europe" .
}
```

Or in **Datalog**, as rule-based decomposition. Note how `within_recursive` defines itself (transitive closure), and `migrated` is built by composing lower-level rules:

```datalog
within_recursive(LocID, PlaceName) :- location(LocID, PlaceName, _).        /* Rule 1 */
within_recursive(LocID, PlaceName) :- within(LocID, ViaID),                 /* Rule 2 */
                                      within_recursive(ViaID, PlaceName).
migrated(PName, BornIn, LivingIn)  :- person(PersonID, PName),              /* Rule 3 */
                                      born_in(PersonID, BornID),
                                      within_recursive(BornID, BornIn),
                                      lives_in(PersonID, LivingID),
                                      within_recursive(LivingID, LivingIn).
us_to_europe(Person)               :- migrated(Person, "United States",     /* Rule 4 */
                                               "Europe").
```

Datalog lets you break complex queries into composable rules the way you break code into functions — especially powerful when recursion and reuse matter.

---

## GraphQL: Client-Driven JSON, Deliberately Limited

GraphQL flips the control: the *client* declares the shape of the JSON it wants, the server resolves it. It's intentionally restricted — **no recursion, no arbitrary search** — because clients are untrusted and DoS risk is real.

```mermaid
flowchart LR
  Client[Mobile / web client] -- "query { channels { name, messages { ... } } }" --> GW[GraphQL gateway]
  GW --> R1[REST / gRPC: channels service]
  GW --> R2[REST / gRPC: messages service]
  GW --> R3[REST / gRPC: users service]
  R1 --> DB1[(channels DB)]
  R2 --> DB2[(messages DB)]
  R3 --> DB3[(users DB)]
  R1 --> GW
  R2 --> GW
  R3 --> GW
  GW -- "JSON shaped exactly as asked" --> Client
```

Example query for a chat app:

```graphql
query ChatApp {
  channels {
    name
    recentMessages(latest: 50) {
      timestamp
      content
      sender { fullName imageUrl }
      replyTo {
        content
        sender { fullName }
      }
    }
  }
}
```

The response mirrors the query shape exactly. Sender names and avatars are **duplicated** inside every message — GraphQL accepts that as the price of simple client rendering, since the alternative (returning IDs only) forces the client into its own N+1 round-trip loop.

---

## Event Sourcing and CQRS

Instead of storing current state, store the **immutable log of events** that produced it. Derive as many read-optimized materialized views as you want.

```mermaid
flowchart LR
  subgraph Commands[Commands - write side]
    C1[OpenRegistrations] --> V[Validator]
    C2[BookSeat] --> V
    C3[CancelBooking] --> V
    C4[ChangeRoomCapacity] --> V
  end

  V -- valid? append --> Log[(Event Log<br/>append-only, ordered)]

  subgraph Views[Materialized Views - read side]
    MV1[bookings_by_attendee<br/>- status, seat, invoice]
    MV2[dashboard_counts<br/>- seats sold, revenue]
    MV3[badge_print_queue<br/>- name, company]
  end

  Log -- replay / subscribe --> MV1
  Log -- replay / subscribe --> MV2
  Log -- replay / subscribe --> MV3

  classDef cmd fill:#efe,stroke:#383
  classDef log fill:#ffd,stroke:#aa0,stroke-width:2px
  classDef view fill:#def,stroke:#37a
  class C1,C2,C3,C4 cmd
  class Log log
  class MV1,MV2,MV3 view
```

**Core properties:**
- Events are named in the **past tense** ("SeatWasBooked"), reflecting facts that happened.
- Commands are validated *before* becoming events; consumers of the log never reject an event.
- Views are **reproducible**: delete one, replay the log, get the same answer (assuming event processing is deterministic).
- CQRS = Command Query Responsibility Segregation: writes go to the log, reads come from views optimized per query pattern.

**Event sourcing vs. star schema** — both are "collections of things that happened":

| Aspect | Fact table (star schema) | Event log (event sourcing) |
|---|---|---|
| Shape | All rows same columns | Heterogeneous event types |
| Order | Unordered (indexed by dims) | Strict order matters |
| Mutability | Corrections happen in bulk | Append-only; corrections are new events |
| Reads | Ad-hoc analytical queries | Specific pre-built materialized views |

**Watch-outs:** GDPR deletion is hard (crypto-shredding is the workaround); external lookups at replay time are nondeterministic (bake exchange rates into the event); reprocessing must not re-trigger side effects like emails.

---

## Trade-offs and Alternatives

| Model | Schema enforcement | Join support | Many-to-many fit | Typical query language | Best workload |
|---|---|---|---|---|---|
| **Relational (SQL)** | Schema-on-write, strict | First-class joins | Excellent (associative tables) | SQL | OLTP + analytics (star schemas) |
| **Document (JSON)** | Schema-on-read, implicit | Weak; `$lookup` in some | Poor for deep links | Mongo aggregation, JSONPath | Self-contained trees, one-to-few |
| **Property graph** | Per-vertex, flexible labels | Traversal-native | Natural (edges are first-class) | Cypher (now ISO GQL) | Variable-depth traversal, social/knowledge graphs |
| **Triple store (RDF)** | Per-triple, URI-scoped | Pattern matching | Natural | SPARQL, Datalog | Linked data, recursive/rule-based queries |
| **Event log (sourcing)** | Per-event-type schema | N/A (reads go to views) | Derived via views | Stream processor + view query | Audit-heavy, evolvable, reversible domains |
| **DataFrame / array** | Column-typed, mutable | Relational-like + reshape | Via wide tables / pivots | pandas, Spark DataFrame API | Analytics, ML feature prep |

### When to pick which

- **One-to-few trees, single-document reads dominate** → document.
- **Structured reporting, well-understood schema, moderate scale** → relational.
- **Write-heavy, "is there a join too expensive at read?" for celebrities** → relational + selective denormalization (Twitter timeline pattern).
- **Read-heavy immutable history, petabyte scale, analyst-driven** → star/snowflake warehouse.
- **Many-to-many dominates, heterogeneous vertex types, variable traversal depth** → property graph or triple store.
- **Client controls UI shape, untrusted callers** → GraphQL at the edge, with any DB behind it.
- **Complex business domain, audit + evolvability + reversibility** → event sourcing + CQRS.
- **Numerical analytics, ML model prep, sparse matrices** → DataFrames and array databases.

---

## Key Takeaways

1. **Pick the data model by the dominant relationship type.** One-to-few → document. Many-to-one with joins → relational. Many-to-many with variable depth → graph. Heterogeneous history → event log.
2. **Normalization is a write/read trade-off, not a moral absolute.** Start normalized; denormalize the slow-changing, hot-read fields only after measurement.
3. **The N+1 query problem is a silent ORM tax.** Replacing 51 round trips with 1 join is often a 50x latency win for free.
4. **"Hydrate at read time" is the scalable compromise.** X/Twitter materializes `(post_id, sender_id)` pairs and hydrates content, likes, and avatars on the read path because those fields change too fast to denormalize across hundreds of millions of timelines.
5. **Star schemas aren't lazy — they reflect the read-dominated, immutable nature of analytical data.** OBT pushes denormalization to the extreme only because analytics tolerates it.
6. **Recursive traversal exposes query-language fitness.** A 4-line Cypher query becomes a 31-line SQL `WITH RECURSIVE`; Datalog decomposes it into composable rules. Language ergonomics translate directly to engineering cost.
7. **GraphQL trades expressiveness for safety**, and accepts response duplication as the price of simple UI rendering against untrusted clients.
8. **Event sourcing decouples write model from read models.** One append-only log, N materialized views — at the cost of making external side effects and personal-data deletion harder to reason about.

---

## See Also

- [[01-relational-vs-document-models]] — schema-on-write vs schema-on-read, locality, the LinkedIn profile tree example
- [[02-normalization-and-denormalization]] — write-vs-read cost, the Twitter timeline hydration pattern, the N+1 query trap
- [[03-star-and-snowflake-schemas]] — fact tables, dimensions, OBT, why analytics tolerates denormalization
- [[04-property-graphs-and-cypher]] — labeled vertices and edges, the Lucy/Idaho/USA/Europe running example, variable-length traversal
- [[05-triple-stores-and-datalog]] — RDF triples, SPARQL pattern matching, Datalog recursive rules and transitive closure
- [[06-graphql]] — client-driven JSON shape, intentional limits against untrusted callers, duplication-for-simplicity
- [[07-event-sourcing-and-cqrs]] — immutable event log as source of truth, CQRS separation of writes and read views